# Understand the Silver Relational Dataset

This notebook introduces the Silver release: the cleaned relational layer over train events, journeys, railway infrastructure, and weather observations. It assumes that the Silver release has already been downloaded under `../data/silver`.

## Setup

In [ ]:
from pathlib import Path

import contextily as ctx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from matplotlib.collections import LineCollection
from matplotlib.legend_handler import HandlerBase
from matplotlib.lines import Line2D

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

silver_dir = Path("../data/silver")

target_train_id = "7572"
target_service_date = "02JAN2023"

silver_dir

## Silver Layout

The Silver release contains partitioned event and journey tables, plus static infrastructure and weather tables.

In [ ]:
expected_tables = {
    "events": silver_dir / "events" / "events_*.parquet",
    "journeys": silver_dir / "journeys" / "journeys_*.parquet",
    "op_nodes": silver_dir / "static" / "op_nodes.parquet",
    "line_sections": silver_dir / "static" / "line_sections.parquet",
    "node_links": silver_dir / "static" / "node_links.parquet",
    "weather": silver_dir / "static" / "weather.parquet",
}

layout = []
for name, path in expected_tables.items():
    if "*" in path.name:
        files = sorted(path.parent.glob(path.name))
        location = str(path)
    else:
        files = [path] if path.exists() else []
        location = str(path)
    layout.append({"table": name, "location": location, "files_found": len(files)})

pd.DataFrame(layout)

If the tables are missing, download the Silver release from the repository root with:

```bash
python -m scripts.dataset.download.download_dataset --target silver
```

## Table Overview

This section inspects table sizes and representative columns without loading every partition into memory.

In [ ]:
def parquet_files(path_pattern: Path) -> list[Path]:
    if "*" in path_pattern.name:
        return sorted(path_pattern.parent.glob(path_pattern.name))
    return [path_pattern] if path_pattern.exists() else []


def parquet_row_count(files: list[Path]) -> int:
    return sum(pq.ParquetFile(path).metadata.num_rows for path in files)


def first_parquet_sample(files: list[Path], n: int = 5) -> pd.DataFrame:
    if not files:
        raise FileNotFoundError("No parquet files found. Download the Silver release first.")
    parquet_file = pq.ParquetFile(files[0])
    if parquet_file.num_row_groups == 0:
        return pd.DataFrame()
    return parquet_file.read_row_group(0).to_pandas().head(n)


overview = []
for name, path in expected_tables.items():
    files = parquet_files(path)
    sample = first_parquet_sample(files, n=1) if files else pd.DataFrame()
    overview.append(
        {
            "table": name,
            "n_files": len(files),
            "n_rows": parquet_row_count(files) if files else None,
            "n_columns": len(sample.columns),
            "columns": ", ".join(sample.columns[:8]),
        }
    )

pd.DataFrame(overview)

## Inspect Individual Tables

In [ ]:
month = pd.to_datetime(target_service_date, format="%d%b%Y").strftime("%Y%m")
journeys_month = pd.read_parquet(silver_dir / "journeys" / f"journeys_{month}.parquet")
events_month = pd.read_parquet(silver_dir / "events" / f"events_{month}.parquet")
op_nodes = pd.read_parquet(expected_tables["op_nodes"])
line_sections = pd.read_parquet(expected_tables["line_sections"])
node_links = pd.read_parquet(expected_tables["node_links"])
weather_sample = first_parquet_sample(parquet_files(expected_tables["weather"]))

The monthly `events` partition contains one row per train event at an operational point, with planned and observed timestamps, event type, delay, and local line context.

In [ ]:
events_month.head()

The matching monthly `journeys` partition summarizes each train service and stores journey-level metadata, aggregate timing information, and inferred graph paths.

In [ ]:
journeys_month.head()

`op_nodes` describes operational points in the railway network, including their identifiers, names, types, and coordinates.

In [ ]:
op_nodes.head()

`line_sections` contains the physical railway line sections used to reconstruct the network geometry.

In [ ]:
line_sections.head()

`node_links` is the graph edge table derived from line sections, where each row connects two consecutive operational points.

In [ ]:
node_links.head()

`weather` stores hourly Open-Meteo observations for operational points; the notebook only samples it here because the full table is large.

In [ ]:
weather_sample

## Relations Between Tables

The main joins are organized around train identifiers, service dates, operational point identifiers, and weather timestamps.

In [ ]:
relations = pd.DataFrame(
    [
        {
            "relation": "journeys -> events",
            "keys": "train_id, service_date",
            "purpose": "retrieve the event sequence of a journey",
        },
        {
            "relation": "events -> op_nodes",
            "keys": "op_id",
            "purpose": "attach operational-point metadata and coordinates to events",
        },
        {
            "relation": "node_links -> line_sections",
            "keys": "line_section_id",
            "purpose": "recover the physical line section that produced a graph edge",
        },
        {
            "relation": "journeys -> node_links",
            "keys": "deduced_paths contains link_id values",
            "purpose": "recover the inferred graph path between successive journey events",
        },
        {
            "relation": "weather -> op_nodes",
            "keys": "op_id",
            "purpose": "attach weather observations to operational points",
        },
    ]
)

with pd.option_context("display.max_colwidth", None):
    display(relations)

## Example: Follow One Journey

The following cells select a fixed example journey from the loaded monthly partition. The chosen journey has 15 events, contains arrival, departure, and pass-through rows, and has short inferred paths between consecutive events.

In [ ]:
journey = journeys_month.query(
    "train_id == @target_train_id and service_date == @target_service_date"
).iloc[0]
journey[["train_id", "service_date", "train_relation", "start_op_id", "end_op_id", "events_count"]]

In [ ]:
journey_train_id = journey["train_id"]
journey_service_date = journey["service_date"]

journey_events = events_month.query(
    "train_id == @journey_train_id and service_date == @journey_service_date"
).sort_values("planned_ts")

journey_events_enriched = journey_events.merge(
    op_nodes[["op_id", "op_name", "lat", "lon"]],
    on="op_id",
    how="left",
).reset_index(drop=True)
journey_events_enriched["weather_time"] = pd.to_datetime(journey_events_enriched["observed_ts"]).dt.floor("h")

path_to_next_event = list(journey["deduced_paths"])
path_to_next_event = path_to_next_event[: len(journey_events_enriched)]
path_to_next_event += [None] * (len(journey_events_enriched) - len(path_to_next_event))
journey_events_enriched["path_to_next_event"] = path_to_next_event

weather_columns = [
    "op_id",
    "time",
    "temperature_2m",
    "rain",
    "snowfall",
    "relative_humidity_2m",
    "wind_speed_10m",
    "weather_code",
]
weather_op_ids = journey_events_enriched["op_id"].astype("int64").unique().tolist()
weather_times = journey_events_enriched["weather_time"].unique().tolist()

weather_for_journey = pd.read_parquet(
    expected_tables["weather"],
    columns=weather_columns,
    filters=[("op_id", "in", weather_op_ids)],
)
weather_for_journey["time"] = pd.to_datetime(weather_for_journey["time"])
weather_for_journey = weather_for_journey.query("time in @weather_times").rename(
    columns={
        "time": "weather_time",
        "temperature_2m": "temperature_2m",
        "rain": "rain",
        "snowfall": "snowfall",
        "relative_humidity_2m": "relative_humidity_2m",
        "wind_speed_10m": "wind_speed_10m",
    }
)

journey_events_enriched = journey_events_enriched.merge(
    weather_for_journey,
    on=["op_id", "weather_time"],
    how="left",
)

The enriched journey event view combines event timing, operational-point metadata, the inferred path to the next event, and weather observed at the event location.

In [ ]:
journey_events_enriched[
    [
        "event_type",
        "op_id",
        "op_name",
        "lat",
        "lon",
        "planned_ts",
        "observed_ts",
        "delay_sec",
        "path_to_next_event",
        "temperature_2m",
        "rain",
        "snowfall",
        "relative_humidity_2m",
        "wind_speed_10m",
        "weather_code",
        "arr_line_id",
        "dep_line_id",
    ]
]

## Visualize the Inferred Journey Path

This lightweight plot shows the selected journey on the local railway graph. Each color corresponds to the inferred path from one event to the next; blue nodes are event locations, dark nodes are nearby operational points, and gray links are the surrounding network.

In [ ]:
node_positions = op_nodes.set_index("op_id")[["lon", "lat"]].astype(float)
node_link_lookup = node_links.set_index("link_id")[["u_node_id", "v_node_id"]].to_dict("index")


class GradientLegendHandle:
    def __init__(self, cmap, label):
        self.cmap = cmap
        self._label = label

    def get_label(self):
        return self._label


class HandlerGradient(HandlerBase):
    def create_artists(self, legend, orig_handle, xdescent, ydescent, width, height, fontsize, trans):
        n_steps = 14
        step_width = width / n_steps
        return [
            plt.Rectangle(
                (xdescent + i * step_width, ydescent + 0.25 * height),
                step_width,
                0.5 * height,
                transform=trans,
                facecolor=orig_handle.cmap(0.05 + 0.90 * i / max(1, n_steps - 1)),
                edgecolor="none",
            )
            for i in range(n_steps)
        ]


def link_segment(link_id):
    link = node_link_lookup.get(int(link_id))
    if link is None:
        return None
    u = int(link["u_node_id"])
    v = int(link["v_node_id"])
    if u not in node_positions.index or v not in node_positions.index:
        return None
    return node_positions.loc[[u, v]].to_numpy()


journey_subpaths = []
for path in journey_events_enriched["path_to_next_event"]:
    if path is None or len(path) == 0:
        journey_subpaths.append([])
        continue
    subpath = []
    for link_id in path:
        segment = link_segment(link_id)
        if segment is not None:
            subpath.append(segment)
    journey_subpaths.append(subpath)

journey_segments = [segment for subpath in journey_subpaths for segment in subpath]
event_nodes = journey_events_enriched.copy()

if not journey_segments or event_nodes.empty:
    raise ValueError("No inferred path segments available for this journey.")

path_coords = np.vstack(journey_segments)
min_lon, min_lat = path_coords.min(axis=0)
max_lon, max_lat = path_coords.max(axis=0)
pad_lon = max(0.08 * (max_lon - min_lon), 0.03)
pad_lat = max(0.10 * (max_lat - min_lat), 0.02)
left = min_lon - pad_lon
right = max_lon + pad_lon
bottom = min_lat - pad_lat
top = max_lat + pad_lat

nearby_nodes = op_nodes.query(
    "@left <= lon <= @right and @bottom <= lat <= @top"
).copy()
event_op_ids = set(event_nodes["op_id"].astype(int))
other_nodes = nearby_nodes.query("op_id not in @event_op_ids")

background_segments = []
for row in node_links.itertuples(index=False):
    segment = link_segment(row.link_id)
    if segment is None:
        continue
    lon = segment[:, 0]
    lat = segment[:, 1]
    if lon.max() < left or lon.min() > right:
        continue
    if lat.max() < bottom or lat.min() > top:
        continue
    background_segments.append(segment)

fig, ax = plt.subplots(figsize=(8, 7), dpi=140)
ax.add_collection(LineCollection(background_segments, colors="#B8C0CC", linewidths=0.8, alpha=0.7, zorder=1))

path_cmap = plt.cm.turbo
colors = path_cmap(np.linspace(0.05, 0.95, max(1, len(journey_subpaths))))
for color, subpath in zip(colors, journey_subpaths):
    if subpath:
        ax.add_collection(LineCollection(subpath, colors=[color], linewidths=2.6, alpha=0.95, zorder=3))

ax.scatter(other_nodes["lon"], other_nodes["lat"], s=8, color="#334155", alpha=0.55, zorder=4)
ax.scatter(event_nodes["lon"], event_nodes["lat"], s=28, color="#2563EB", edgecolors="white", linewidths=0.5, zorder=5)
ax.scatter(event_nodes.iloc[0]["lon"], event_nodes.iloc[0]["lat"], s=48, color="#16A34A", edgecolors="white", linewidths=0.8, zorder=6)
ax.scatter(event_nodes.iloc[-1]["lon"], event_nodes.iloc[-1]["lat"], s=48, color="#DC2626", edgecolors="white", linewidths=0.8, zorder=6)

ax.set_xlim(left, right)
ax.set_ylim(bottom, top)
ax.set_aspect(1 / np.cos(np.deg2rad(path_coords[:, 1].mean())), adjustable="box")
try:
    ctx.add_basemap(ax, crs="EPSG:4326", source=ctx.providers.CartoDB.PositronNoLabels, zorder=0)
    ctx.add_basemap(ax, crs="EPSG:4326", source=ctx.providers.CartoDB.PositronOnlyLabels, alpha=0.7, zorder=2)
except Exception as exc:
    print(f"Basemap unavailable: {exc}")
ax.axis("off")
ax.legend(
    handles=[
        GradientLegendHandle(path_cmap, "path"),
        Line2D([], [], color="#B8C0CC", linewidth=1.0, label="other node links"),
        Line2D([], [], marker="o", linestyle="None", markersize=4, color="#334155", markerfacecolor="#334155", label="other nodes"),
        Line2D([], [], marker="o", linestyle="None", markersize=4.5, color="#2563EB", markerfacecolor="#2563EB", markeredgecolor="white", label="event nodes"),
        Line2D([], [], marker="o", linestyle="None", markersize=5, color="#16A34A", markerfacecolor="#16A34A", markeredgecolor="white", label="start event node"),
        Line2D([], [], marker="o", linestyle="None", markersize=5, color="#DC2626", markerfacecolor="#DC2626", markeredgecolor="white", label="end event node"),
    ],
    loc="upper left",
    fontsize=8,
    handlelength=1.8,
    labelspacing=0.35,
    borderpad=0.45,
    frameon=True,
    framealpha=0.92,
    handler_map={GradientLegendHandle: HandlerGradient()},
)
plt.show()

[Back to README](../README.md#what-do-you-want-to-do)